In [1]:
import polars as pl

df = pl.read_parquet("trigrams.parquet")

counts = (
    df.group_by(["word1", "word2", "word3"])
      .len()
      .rename({"len": "count"})
)

counts.head()

word1,word2,word3,count
str,str,str,u32
"""appraiser""","""'""","""s""",5
"""his""","""wife""","""embezzled""",5
"""guidance""","""for""","""earnings""",1
"""cheered""","""a""","""big""",1
"""iraqi""","""justice""","""rather""",1


In [2]:
totals = (
    counts.group_by(["word1", "word2"])
          .agg(pl.sum("count").alias("total"))
)

model = counts.join(totals, on=["word1","word2"])

model = model.with_columns(
    (pl.col("count") / pl.col("total")).alias("prob")
)

model.head()

word1,word2,word3,count,total,prob
str,str,str,u32,u32,f64
"""appraiser""","""'""","""s""",5,5,1.0
"""his""","""wife""","""embezzled""",5,8132,0.000615
"""guidance""","""for""","""earnings""",1,111,0.009009
"""cheered""","""a""","""big""",1,76,0.013158
"""iraqi""","""justice""","""rather""",1,22,0.045455


In [3]:
from collections import defaultdict

lookup = defaultdict(list)

for row in model.iter_rows(named=True):
    lookup[(row["word1"], row["word2"])].append(
        (row["word3"], row["prob"])
    )

lookup[("his", "wife")]

[('embezzled', 0.0006148548942449582),
 ('completed', 0.00012297097884899163),
 ('bun', 0.00024594195769798326),
 ('hsu', 0.00012297097884899163),
 ('princess', 0.002336448598130841),
 ('iman', 0.00012297097884899163),
 ('promises', 0.00012297097884899163),
 ('martine', 0.00012297097884899163),
 ('suspecting', 0.00012297097884899163),
 ('looked', 0.00012297097884899163),
 ('ruth', 0.0006148548942449582),
 ('picked', 0.00012297097884899163),
 ('aboard', 0.00024594195769798326),
 ('sliced', 0.00012297097884899163),
 ('now', 0.00024594195769798326),
 ('contributed', 0.00012297097884899163),
 ('deng', 0.0004918839153959665),
 ('sybille', 0.00012297097884899163),
 ('home', 0.00024594195769798326),
 ('scream', 0.00024594195769798326),
 ('cover', 0.00012297097884899163),
 ('added', 0.00012297097884899163),
 ('during', 0.007009345794392523),
 ('marisol', 0.00012297097884899163),
 ('reported', 0.00012297097884899163),
 ('tried', 0.0004918839153959665),
 ('warmed', 0.00012297097884899163),
 ('ha

In [4]:
for key in lookup:
    lookup[key].sort(key=lambda x: x[1], reverse=True)

In [6]:
import pickle

with open("trigram_model.pkl", "wb") as f:
    pickle.dump(lookup, f)

In [43]:
import pandas as pd
from openai import OpenAI
import re
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
load_dotenv()
import os
from transformers import pipeline

In [ ]:
MODEL = "openai-community/gpt2"

# Load GPT-2 locally
generator = pipeline(
    "text-generation",
    model=MODEL,
    tokenizer=MODEL,
)

# Load dataset
df = pd.read_csv("dev_set.csv")

MAX_ROWS = 1000
df = df.iloc[:MAX_ROWS]

correct = 0
predictions = []

total = len(df)

for i, (_, row) in enumerate(df.iterrows(), start=1):

    # Use only the context as the prompt
    prompt = row["context"]

    response = generator(
        prompt,
        max_new_tokens=1,            # predict the next token
        do_sample=False,             # greedy decoding
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id,
    )

    generated_text = response[0]["generated_text"].strip()

    print("========== RAW ==========")
    print(generated_text)
    print("=========================")

    # Extract the first generated word
    match = re.search(r"[\w'-]+", generated_text.lower())
    prediction = match.group(0) if match else "[UNK]"

    target = str(row["answer"]).strip().lower()

    is_correct = prediction == target
    correct += is_correct

    predictions.append({
        "context": row["context"],
        "target": target,
        "prediction": prediction,
        "correct": is_correct,
    })

    print(f"{i}/{total}")
    print(f"Target: {target}")
    print(f"Pred:   {prediction}")

accuracy = correct / total

print("=" * 50)
print(f"Accuracy: {accuracy:.4%} ({correct}/{total})")

pd.DataFrame(predictions).to_csv("predictions.csv", index=False)

Device set to use mps:0


========== RAW ==========
year
1/1000
Target: day
Pred:   year
========== RAW ==========
the
2/1000
Target: the
Pred:   the
========== RAW ==========
.
3/1000
Target: morning
Pred:   [UNK]
========== RAW ==========
was
4/1000
Target: was
Pred:   was
========== RAW ==========
to
5/1000
Target: to
Pred:   to
========== RAW ==========
the
6/1000
Target: a
Pred:   the
========== RAW ==========
Bel
7/1000
Target: moscow
Pred:   bel
========== RAW ==========
that
8/1000
Target: that
Pred:   that
========== RAW ==========
of
9/1000
Target: of
Pred:   of
========== RAW ==========
parties
10/1000
Target: women
Pred:   parties
========== RAW ==========
100
11/1000
Target: 11
Pred:   100
========== RAW ==========
and
12/1000
Target: ,
Pred:   and
========== RAW ==========
.
13/1000
Target: ,
Pred:   [UNK]
========== RAW ==========
of
14/1000
Target: tonnes
Pred:   of
========== RAW ==========
the
15/1000
Target: his
Pred:   the
========== RAW ==========
commerce
16/1000
Target: deputies
Pred:   c

In [ ]:
# Load masked language model
fill_mask = pipeline(
    "fill-mask",
    model="FacebookAI/roberta-large",
    tokenizer="FacebookAI/roberta-large",
    )

MASK = fill_mask.tokenizer.mask_token

# Load dataset
df = pd.read_csv("dev_set.csv")

MAX_ROWS = 1000
df = df.iloc[:MAX_ROWS]

correct = 0
predictions = []

TOP_K = 100

for i, row in df.iterrows():

    context = row["context"]
    first_letter = str(row["first letter"]).lower()
    target = str(row["answer"]).strip().lower()

    text = context + " " + MASK

    results = fill_mask(text, top_k=TOP_K)

    prediction = "[UNK]"

    for candidate in results:

        token = candidate["token_str"].strip().lower()

        if token.startswith(first_letter):
            prediction = token
            break

    is_correct = prediction == target
    correct += is_correct

    predictions.append({
        "context": context,
        "first_letter": first_letter,
        "target": target,
        "prediction": prediction,
        "correct": is_correct,
    })

    print(f"{i+1}/{len(df)}")
    print(f"Target: {target}")
    print(f"Pred:   {prediction}")
    print("✓" if is_correct else "✗")
    print()

accuracy = correct / len(df)

print("=" * 50)
print(f"Accuracy: {accuracy:.4%} ({correct}/{len(df)})")

pd.DataFrame(predictions).to_csv("predictions.csv", index=False)

Device set to use mps:0


1/1000
Target: day
Pred:   day
✓

2/1000
Target: the
Pred:   the
✓

3/1000
Target: morning
Pred:   morning
✓

4/1000
Target: was
Pred:   was
✓

5/1000
Target: to
Pred:   to
✓

6/1000
Target: a
Pred:   afghanistan
✗

7/1000
Target: moscow
Pred:   moscow
✓

8/1000
Target: that
Pred:   to
✗

9/1000
Target: of
Pred:   of
✓

10/1000
Target: women
Pred:   women
✓

11/1000
Target: 11
Pred:   15
✗

12/1000
Target: ,
Pred:   ,
✓

13/1000
Target: ,
Pred:   ,
✓

14/1000
Target: tonnes
Pred:   tuna
✗

15/1000
Target: his
Pred:   him
✗

16/1000
Target: deputies
Pred:   deputies
✓

17/1000
Target: shows
Pred:   shows
✓

18/1000
Target: kilometers
Pred:   km
✗

19/1000
Target: palestinian
Pred:   palestinians
✗

20/1000
Target: vests
Pred:   vest
✗

21/1000
Target: the
Pred:   trouble
✗

22/1000
Target: internationals
Pred:   international
✗

23/1000
Target: relatives
Pred:   relatives
✓

24/1000
Target: cup
Pred:   cup
✓

25/1000
Target: for
Pred:   for
✓

26/1000
Target: american
Pred:   airport
✗
